# `sp_score` returns wrong scores when a sequence is marked as a duplicate

`cogent3==2026.9.10`. `get_app("sp_score")` takes its similarity term from the fast pairwise
distance calculator in `cogent3.evolve.fast_distance`. That calculator marks a sequence as a
duplicate of an earlier one whenever the pair has no mismatching site, and then copies the
earlier sequence's statistics to it. But "no mismatches" does not mean "identical": a pair with
no sites in common (one sequence is all gaps or all `N`), or with only a few sites in common
that happen to agree, is also marked as a duplicate. In those situations, and also when two
sequences really are identical, `sp_score` returns the wrong value.

This notebook is self-contained. It describes the mechanism, gives a minimal example for each
situation, and simulates how often the problem occurs for three primate-like sequences.

**Contents**

1. What `sp_score` computes
2. Where the bug is
3. Minimal examples
4. How often it happens: simulation
5. Suggested fix

## 1. What `sp_score` computes

For each pair of sequences let $L$ be the number of sites where both have a nucleotide (gaps
and `N` are excluded), $p$ the proportion of those sites where the nucleotides differ, and $d$
the genetic distance (`pdist`: $d = p$; `JC69`: $d = -\tfrac34\ln(1 - \tfrac43 p)$). The app
computes for each pair

$$\text{similarity} = \max\bigl(L\,(1 - d),\ 0\bigr),$$

subtracts an affine gap penalty (`gap_insert` for every gap run present in only one of the two
sequences, plus `gap_extend` for each of its positions), and sums over all pairs.

`expected_sp_score` below computes exactly this from the sequences. It reuses the app's own
gap-penalty app, so any difference from `sp_score` is in the similarity term. The examples use
`pdist` so the numbers can be checked by hand; the default `JC69` is reported as well. With the
default penalties, an all-gap sequence of length $n$ costs $12 + n$ against each sequence
without gaps.

In [1]:
from itertools import combinations

import numpy as np
import pandas as pd

from cogent3 import make_aligned_seqs, get_app
from cogent3.evolve.fast_distance import get_distance_calculator

GAP_INSERT, GAP_EXTEND = 12.0, 1.0
BASES = list("ACGT")
APPS = {
    calc: get_app("sp_score", calc=calc, gap_insert=GAP_INSERT, gap_extend=GAP_EXTEND)
    for calc in ("pdist", "JC69")
}


def expected_sp_score(aln, calc="pdist"):
    """sum-of-pairs score computed directly from the sequences"""
    arr = {n: np.array(list(str(aln.seqs[n]))) for n in aln.names}
    gaps = APPS[calc]._gap_calc(aln)
    total = 0.0
    for a, b in combinations(aln.names, 2):
        both = np.isin(arr[a], BASES) & np.isin(arr[b], BASES)
        length = both.sum()
        if length:
            p = (arr[a][both] != arr[b][both]).mean()
            if calc == "pdist":
                d = p
            elif p < 0.75:
                d = -0.75 * np.log(1 - 4 * p / 3)
            else:
                return np.nan  # JC69 is undefined; the app returns NotCompleted
            total += max(length * (1 - d), 0.0)
        total -= gaps[a, b]
    return total


def check(seqs):
    """show the alignment, the duplicates the calculator detects, app vs expected score"""
    aln = make_aligned_seqs(seqs, moltype="dna")
    width = max(map(len, seqs))
    for name, seq in seqs.items():
        print(f"{name:<{width}}  {seq}")
    rows = {}
    for name, app in APPS.items():
        calc = get_distance_calculator(name)
        calc(aln, show_progress=False)
        result = app(aln)
        rows[name] = {
            "duplicates detected": calc.duplicated or "none",
            "sp_score app": float(result) if isinstance(result, float) else "NotCompleted",
            "expected": expected_sp_score(aln, name),
        }
    return pd.DataFrame(rows).T

## 2. Where the bug is

`_PairwiseDistance.run` in `cogent3/evolve/fast_distance.py` visits the pairs in sequence
order: (1st, 2nd), (1st, 3rd), …, (2nd, 3rd), …. For each pair it fills a matrix counting the
nucleotide pairs at the sites where both sequences have a nucleotide. If no off-diagonal cell
is filled, i.e. the pair has no mismatching site, the later sequence is recorded as a duplicate
of the earlier one and is skipped in every later comparison:

```python
fill_diversity_matrix(matrix, s1, s2)
if not (matrix[off_diag] > 0).any():
    # j is a duplicate of i
    dupes.update([j])
    duped[i].append(j)
    continue
```

`_expand` then fills in the duplicate's statistics (`dists`, `lengths`, …): 0 against the
sequence it duplicates, and a copy of that sequence's values against every other sequence:

```python
val = 0 if name == alias else pwise.get((alias, name), None)
pwise[(add, name)] = pwise[(name, add)] = val
```

Two things go wrong:

- **No mismatches, but not identical.** A sequence with no sites in common with an earlier one
  (all gaps, or all `N` for the model-based distances), or with only a few shared sites that all
  agree, is treated as a copy of that sequence. It is given distances and aligned lengths it
  does not have, so `sp_score` credits similarity that is not in the alignment. If the sparse
  sequence comes first, the later sequences are never compared with each other and their
  similarity is lost.
- **Identical sequences.** The aligned length of the (original, duplicate) pair is set to 0, so
  `sp_score` gives a similarity of 0 to the pair that should score highest.

Which case occurs depends on the order of the sequences in the alignment.

## 3. Minimal examples

All examples start from the same 10-column alignment. Chimpanzee differs from Human at
column 9 only; Orangutan differs from Human at column 4, and from Chimpanzee at columns 4 and 9.

```
column      1234567890
Human       ACGTACGTAC
Chimpanzee  ACGTACGTTC
Orangutan   ACGAACGTAC
```

With `pdist` the pairwise similarities are Human–Chimpanzee 9, Human–Orangutan 9 and
Chimpanzee–Orangutan 8, so the score is 26. The app agrees when nothing is marked as a
duplicate.

In [2]:
HUMAN = "ACGTACGTAC"
CHIMP = "ACGTACGTTC"
ORANG = "ACGAACGTAC"
N = len(HUMAN)

check({"Human": HUMAN, "Chimpanzee": CHIMP, "Orangutan": ORANG})

Human       ACGTACGTAC
Chimpanzee  ACGTACGTTC
Orangutan   ACGAACGTAC


,duplicates detected,sp_score app,expected
pdist,none,26.0,26.0
JC69,none,25.527325,25.527325


### Case 1: a later sequence is entirely gaps

**1a. Orangutan is all gaps.** Human and Orangutan have no site in common, so Orangutan is
marked as a duplicate of Human and inherits Human's distance to Chimpanzee (0.1 over 10 sites).

Expected: $9 - 2\,(12 + 10) = -35$. The app also credits Chimpanzee–Orangutan with 9:
$9 + 9 - 44 = -26$.

In [3]:
check({"Human": HUMAN, "Chimpanzee": CHIMP, "Orangutan": "-" * N})

Human       ACGTACGTAC
Chimpanzee  ACGTACGTTC
Orangutan   ----------


,duplicates detected,sp_score app,expected
pdist,{'Human': ['Orangutan']},-26.0,-35.0
JC69,{'Human': ['Orangutan']},-26.146513,-35.073256


**1b. Chimpanzee is all gaps.** Chimpanzee becomes a duplicate of Human, and the pair
Chimpanzee–Orangutan is credited with Human–Orangutan's similarity of 9.

Expected: $9 - 44 = -35$. App: $9 + 9 - 44 = -26$.

In [4]:
check({"Human": HUMAN, "Chimpanzee": "-" * N, "Orangutan": ORANG})

Human       ACGTACGTAC
Chimpanzee  ----------
Orangutan   ACGAACGTAC


,duplicates detected,sp_score app,expected
pdist,{'Human': ['Chimpanzee']},-26.0,-35.0
JC69,{'Human': ['Chimpanzee']},-26.146513,-35.073256


### Case 2: a later sequence is entirely `N` (model-based distances)

`JC69` and the other model-based calculators skip `N`, so this is case 1 without gap
penalties: no shared sites, hence no mismatches, and Orangutan becomes a duplicate of Human.

JC69 expected: $10\,(1 - d_{JC}(0.1)) = 8.93$. App: $2 \times 8.93 = 17.85$.

`pdist` is not affected here because its calculator is built on the `text` molecular type,
where `N` is an ordinary character: `N` against `A` counts as a mismatch and `N` against `N` as
a match. That is a different convention rather than a fix (a sequence `ACGNNNNNNN` gets a
pdist of 0.7 to Human over 10 sites instead of 0 over 3), so the `pdist` and `JC69` rows are
only comparable when there is no `N`.

In [5]:
check({"Human": HUMAN, "Chimpanzee": CHIMP, "Orangutan": "N" * N})

Human       ACGTACGTAC
Chimpanzee  ACGTACGTTC
Orangutan   NNNNNNNNNN


,duplicates detected,sp_score app,expected
pdist,none,9.0,9.0
JC69,{'Human': ['Orangutan']},17.853487,8.926744


### Case 3: a later sequence has a few nucleotides that all agree with an earlier sequence

**3a. Orangutan keeps 3 nucleotides matching Human.** Orangutan is marked as a copy of Human,
so Chimpanzee–Orangutan is credited with Human's 10-site similarity to Chimpanzee, and
Human–Orangutan gets an aligned length of 0. Orangutan has one gap run of 7, so the gap
penalty is $12 + 7 = 19$ against each other sequence.

Expected: $9 + 3 + 3 - 2 \times 19 = -23$. App: $9 + 0 + 9 - 38 = -20$.

In [6]:
check({"Human": HUMAN, "Chimpanzee": CHIMP, "Orangutan": "ACG" + "-" * 7})

Human       ACGTACGTAC
Chimpanzee  ACGTACGTTC
Orangutan   ACG-------


,duplicates detected,sp_score app,expected
pdist,{'Human': ['Orangutan']},-20.0,-23.0
JC69,{'Human': ['Orangutan']},-20.146513,-23.073256


**3b. Orangutan keeps 1 nucleotide that differs from Human but matches Chimpanzee.**
Human–Orangutan has a mismatch, so no duplicate there; Chimpanzee–Orangutan has none, so
Orangutan becomes a duplicate of **Chimpanzee** and inherits Chimpanzee's distance to Human
(0.1 over 10 sites) where the truth is 1 mismatch in 1 site. Orangutan has two gap runs
(8 and 1), so the gap penalty is $24 + 9 = 33$.

Expected: $9 + 0 + 1 - 66 = -56$. App: $9 + 9 + 0 - 66 = -48$.

With `JC69` this example shows a second effect. Human–Orangutan has $p = 1$, for which JC69 is
undefined, so the app should return `NotCompleted` (and `expected_sp_score` gives `nan`).
Instead, once Orangutan is marked as a duplicate, `run` deletes every stored result involving
it, including the undefined one, and `_expand` replaces it with a copy of Chimpanzee–Human. The
app returns a number.

In [7]:
check({"Human": HUMAN, "Chimpanzee": CHIMP, "Orangutan": "-" * 8 + "T-"})

Human       ACGTACGTAC
Chimpanzee  ACGTACGTTC
Orangutan   --------T-


,duplicates detected,sp_score app,expected
pdist,{'Chimpanzee': ['Orangutan']},-48.0,-56.0
JC69,{'Chimpanzee': ['Orangutan']},-48.146513,NaN


### Case 4: the first sequence has few nucleotides (or none)

The first sequence is never marked as a duplicate itself, but every later sequence with no
mismatch against it is. Those later sequences are then never compared with each other, so
their similarity is lost and the score is **too low**.

**4a. Human keeps 3 nucleotides.** Chimpanzee and Orangutan both agree with Human at columns
1–3, so both become duplicates of Human and Chimpanzee–Orangutan is never computed.

Expected: $3 + 3 + 8 - 2 \times 19 = -24$. App: $0 + 0 + 0 - 38 = -38$.

In [8]:
check({"Human": "ACG" + "-" * 7, "Chimpanzee": CHIMP, "Orangutan": ORANG})

Human       ACG-------
Chimpanzee  ACGTACGTTC
Orangutan   ACGAACGTAC


,duplicates detected,sp_score app,expected
pdist,"{'Human': ['Chimpanzee', 'Orangutan']}",-38.0,-24.0
JC69,"{'Human': ['Chimpanzee', 'Orangutan']}",-38.0,-24.326162


**4b. Human is all gaps.** The same happens with no shared sites at all.

Expected: $8 - 44 = -36$. App: $0 - 44 = -44$.

In [9]:
check({"Human": "-" * N, "Chimpanzee": CHIMP, "Orangutan": ORANG})

Human       ----------
Chimpanzee  ACGTACGTTC
Orangutan   ACGAACGTAC


,duplicates detected,sp_score app,expected
pdist,"{'Human': ['Chimpanzee', 'Orangutan']}",-44.0,-36.0
JC69,"{'Human': ['Chimpanzee', 'Orangutan']}",-44.0,-36.326162


### Case 5: two sequences really are identical

Here the duplicate shortcut is legitimate, but `_expand` sets the aligned length of the
(original, duplicate) pair to 0, so its similarity is dropped.

**Chimpanzee identical to Human.** Expected: $10 + 9 + 9 = 28$. App: $0 + 9 + 9 = 18$, too low
by exactly the 10 shared sites.

In [10]:
check({"Human": HUMAN, "Chimpanzee": HUMAN, "Orangutan": ORANG})

Human       ACGTACGTAC
Chimpanzee  ACGTACGTAC
Orangutan   ACGAACGTAC


,duplicates detected,sp_score app,expected
pdist,{'Human': ['Chimpanzee']},18.0,28.0
JC69,{'Human': ['Chimpanzee']},17.853487,27.853487


### Control: one mismatch is enough to avoid the bug

Take case 3a and change Orangutan's third nucleotide so it differs from both other sequences.
Every pair now has a mismatch, nothing is marked as a duplicate, and the app is correct.

Expected and app: $9 + 2 + 2 - 38 = -25$.

(Separately, a pair with $p \ge 0.75$ has no `JC69` distance and the app returns
`NotCompleted`. That is a documented limitation, not this bug; `expected_sp_score` returns
`nan` in that case.)

In [11]:
check({"Human": HUMAN, "Chimpanzee": CHIMP, "Orangutan": "ACT" + "-" * 7})

Human       ACGTACGTAC
Chimpanzee  ACGTACGTTC
Orangutan   ACT-------


,duplicates detected,sp_score app,expected
pdist,none,-25.0,-25.0
JC69,none,-25.718296,-25.718296


### Summary

| Case | Situation | Error in `sp_score` |
| --- | --- | --- |
| 1 | a later sequence entirely gaps | too high: copied similarity |
| 2 | a later sequence entirely `N` (model-based distances) | too high: copied similarity |
| 3 | a later sequence with a few nucleotides, all agreeing with an earlier one | too high: copied full-length similarity |
| 4 | the first sequence sparse or empty | too low: later sequences never compared |
| 5 | two sequences identical | too low: the identical pair's similarity dropped |

In cases 1–4 the trigger is the same: a pair with **no mismatch among its shared sites** that
is not a pair of identical sequences. The fewer sites a pair shares, the more likely this is.

## 4. How often it happens: simulation

A 1 kb alignment of three sequences: Chimpanzee 1.2% and Orangutan 3.1% diverged from Human.
One of the three keeps only 100 nucleotides, either as a single 100 bp block or as 100 scattered
columns, and is gaps elsewhere. An alignment counts as wrong if the app and `expected_sp_score`
differ. 300 replicates per setting with `pdist` (about a minute).

In [12]:
rng = np.random.default_rng(7)
WINDOW, KEEP, REPS = 1_000, 100, 300


def mutate(seq, rate):
    seq = seq.copy()
    hit = rng.random(len(seq)) < rate
    seq[hit] = [BASES[(BASES.index(b) + rng.integers(1, 4)) % 4] for b in seq[hit]]
    return seq


rows = []
for sparse in ("Human", "Chimpanzee", "Orangutan"):
    for layout in ("one 100 bp block", "100 scattered columns"):
        errors = []
        for _ in range(REPS):
            human = rng.choice(BASES, WINDOW)
            seqs = {"Human": human, "Chimpanzee": mutate(human, 0.012), "Orangutan": mutate(human, 0.031)}
            if layout.startswith("one"):
                start = rng.integers(0, WINDOW - KEEP + 1)
                keep = np.arange(start, start + KEEP)
            else:
                keep = rng.choice(WINDOW, KEEP, replace=False)
            thinned = np.full(WINDOW, "-")
            thinned[keep] = seqs[sparse][keep]
            seqs[sparse] = thinned
            aln = make_aligned_seqs({k: "".join(v) for k, v in seqs.items()}, moltype="dna")
            errors.append(float(APPS["pdist"](aln)) - expected_sp_score(aln, "pdist"))
        errors = np.array(errors)
        wrong = np.abs(errors) > 1e-6
        rows.append(
            {
                "sparse sequence": sparse,
                "layout": layout,
                "fraction wrong": wrong.mean(),
                "mean error when wrong": errors[wrong].mean() if wrong.any() else 0.0,
            }
        )

pd.DataFrame(rows).round(3)

,sparse sequence,layout,fraction wrong,mean error when wrong
0,Human,one 100 bp block,0.333,-971.180
1,Human,100 scattered columns,0.323,-970.052
2,Chimpanzee,one 100 bp block,0.260,770.808
3,Chimpanzee,100 scattered columns,0.293,772.409
4,Orangutan,one 100 bp block,0.040,786.750
5,Orangutan,100 scattered columns,0.063,789.158


A sequence is marked as a duplicate when the sites it shares with an earlier sequence contain
no mismatch. With 100 shared sites that happens with probability $0.988^{100} \approx 0.30$ for
Chimpanzee against Human and $0.969^{100} \approx 0.04$ for Orangutan against Human, plus a
smaller contribution from the Chimpanzee–Orangutan pair ($\approx 0.957^{100} \approx 0.01$).
That matches the table: about 30% when Chimpanzee is sparse, 4–6% when Orangutan is sparse,
and about 33% when Human is sparse (Chimpanzee or Orangutan becomes a copy of Human, case 4).

When an alignment is hit, the error is the size of one pairwise similarity. With Chimpanzee or
Orangutan sparse it is about $+770$: the copied full-length similarity ($\approx 970$) minus the
sparse sequence's two true similarities ($\approx 100 + 97$). With Human sparse it is about
$-970$: the Chimpanzee–Orangutan similarity that is never computed.

## 5. Suggested fix

- In `_PairwiseDistance.run`, record `j` as a duplicate of `i` only when the two index arrays
  are equal at every position (for example `numpy.array_equal(s1, s2)`), not when the diversity
  matrix merely has an empty off-diagonal. Alternatively drop the shortcut and call `self.func`
  for every pair.
- If the shortcut is kept, `_expand` should give the (original, duplicate) pair its real
  statistics (aligned length = number of shared sites, distance 0, …) rather than 0 for every
  statistic, so that `lengths` is correct and `sp_score` scores the identical pair.

With either change all the examples above give the expected score.